# TP-1: CleaningData - Applying OOP and Design Patterns

Each exercise builds upon the previous one - so by the end, we will have a complete CSV 
cleaning data using OOP, decorators, and patterns.


## Exercise 1: Build the CSVReader Class (OOP Foundation)

### Objective:
- Reinforce OOP fundamentals: classes, constructors, methods, encapsulation.
- Learn to design reusable data classes for DS projects.

### Concept:
A CSVReader class should encapsulate all functionality related to reading CSV files.

### Instructions:
1. Create a class CSVReader.
2. Define an __init__ method that takes file_path.
3. Add a read() method that loads the CSV file using pandas.
4. Add a preview(n) method that prints the top n rows.


In [1]:
import pandas as pd


class CSVReader:
    """Read and preview data stored in a CSV file."""

    def __init__(self, file_path):
        self.file_path = file_path
        self.data = None

    def read(self):
        """Load the CSV file and return it as a pandas DataFrame."""
        self.data = pd.read_csv(self.file_path)
        return self.data

    def preview(self, n=5):
        """Print the first *n* rows of the CSV file."""
        if self.data is None:
            self.read()
        print(self.data.head(n))


# Example usage
reader = CSVReader("sample_data (2).csv")
data = reader.read()
reader.preview(5)


   id     name   age  height_cm  weight_kg         city  score
0   1    Alice  29.0      165.0       68.0     New York   85.0
1   2      Bob   NaN      172.0        NaN  Los Angeles   90.0
2   3  Charlie  35.0      168.0       72.0      Chicago    NaN
3   4    David   NaN        NaN       80.0      Houston   75.0
4   5      Eva  27.0      160.0       55.0     New York   88.0


## Exercise 2: Apply Pattern for Missing Values Handling

### Objective:
- Learn the Strategy Pattern - one interface, many interchangeable behaviors.
- Practice inheritance and polymorphism.

## Concept:
We often need different strategies to handle missing data (drop, fill mean, fill mode, etc.). The pattern allows flexible switching between methods `without changing the main code`.

## Instructions:

1. Create an abstract class MissingValueStrategy with a method handle(df).

2. Create subclasses:
- DropMissing
- FillMean
- FillMode

3. Create a DataCleaner class that applies the chosen strategy.



In [2]:
from abc import ABC, abstractmethod


class MissingValueStrategy(ABC):
    """Interface for interchangeable missing-value handling strategies."""

    @abstractmethod
    def handle(self, df):
        """Return a DataFrame with missing values handled."""


class DropMissing(MissingValueStrategy):
    """Remove every row containing at least one missing value."""

    def handle(self, df):
        return df.dropna().reset_index(drop=True)


class FillMean(MissingValueStrategy):
    """Fill missing numeric values with the mean of their column."""

    def handle(self, df):
        result = df.copy()
        numeric_columns = result.select_dtypes(include="number").columns
        result[numeric_columns] = result[numeric_columns].fillna(
            result[numeric_columns].mean()
        )
        return result


class FillMode(MissingValueStrategy):
    """Fill missing values with the most common value in each column."""

    def handle(self, df):
        result = df.copy()
        for column in result.columns:
            modes = result[column].mode(dropna=True)
            if not modes.empty:
                result[column] = result[column].fillna(modes.iloc[0])
        return result


class DataCleaner:
    """Apply a selected missing-value strategy to a DataFrame."""

    def __init__(self, strategy):
        if not isinstance(strategy, MissingValueStrategy):
            raise TypeError("strategy must be a MissingValueStrategy")
        self.strategy = strategy

    def clean(self, df):
        return self.strategy.handle(df)


# Example usage: switch strategies without changing DataCleaner.
raw_data = reader.read()
mean_cleaner = DataCleaner(FillMean())
mean_data = mean_cleaner.clean(raw_data)
print(mean_data)


   id     name    age  height_cm  weight_kg         city  score
0   1    Alice  29.00      165.0       68.0     New York   85.0
1   2      Bob  30.25      172.0       70.0  Los Angeles   90.0
2   3  Charlie  35.00      168.0       72.0      Chicago   86.0
3   4    David  30.25      167.0       80.0      Houston   75.0
4   5      Eva  27.00      160.0       55.0     New York   88.0
5   6    Frank  30.00      175.0       85.0  Los Angeles   86.0
6   7    Grace  30.25      162.0       60.0      Chicago   92.0


## Exercise 3: Add Decorators for Logging and Timing

### Objective:
- Learn the Decorator Pattern to add reusable functionality (logging, timing).
- Understand how decorators support the “Open/Closed Principle”.

### Concept:
Decorators wrap functions to extend their behavior without altering the original code.

### Instructions:
1. Create two decorators:
- @log_action → logs when a method starts and finishes.
- @log_time → calculates and prints execution time.
2. Apply them to methods in CSVReader or DataCleaner.

In [ ]:
from functools import wraps
from time import perf_counter


def log_action(func):
    """Log when a function starts and finishes."""
    @wraps(func)
    def wrapper(*args, **kwargs):
        print(f"Starting {func.__name__}...")
        result = func(*args, **kwargs)
        print(f"Finished {func.__name__}.")
        return result

    return wrapper


def log_time(func):
    """Print how long a function takes to execute."""
    @wraps(func)
    def wrapper(*args, **kwargs):
        start = perf_counter()
        result = func(*args, **kwargs)
        elapsed = perf_counter() - start
        print(f"{func.__name__} took {elapsed:.6f} seconds.")
        return result

    return wrapper


# Decorate existing methods without changing their original implementations.
CSVReader.read = log_time(log_action(CSVReader.read))
DataCleaner.clean = log_time(log_action(DataCleaner.clean))


# Example usage
decorated_reader = CSVReader("sample_data (2).csv")
decorated_data = decorated_reader.read()
decorated_cleaner = DataCleaner(FillMean())
decorated_clean_data = decorated_cleaner.clean(decorated_data)

Starting read...
Finished read.
read took 0.001644 seconds.
Starting clean...
Finished clean.
clean took 0.002121 seconds.


## Exercise 4: Implement Factory Pattern for Data Transformations

### Objective:
- Practice Factory Pattern for scalable creation of transformation objects.
- Apply abstraction and composition.

### Concept:
Instead of manually creating objects, use a factory that decides what transformation to apply.

### Instructions:
1. Create an abstract class DataTransform with a method apply(df).
2. Implement subclasses:
- NormalizeColumns
- RemoveDuplicates
- StandardizeText
3. Create a TransformFactory that returns transformation objects based on string input.

In [4]:
from abc import ABC, abstractmethod


class DataTransform(ABC):
    """Interface for DataFrame transformations."""

    @abstractmethod
    def apply(self, df):
        """Return a transformed copy of the DataFrame."""


class NormalizeColumns(DataTransform):
    """Scale numeric columns to the range 0 to 1."""

    def apply(self, df):
        result = df.copy()
        numeric_columns = result.select_dtypes(include="number").columns
        for column in numeric_columns:
            minimum = result[column].min()
            maximum = result[column].max()
            if maximum != minimum:
                result[column] = (result[column] - minimum) / (maximum - minimum)
            else:
                result[column] = 0.0
        return result


class RemoveDuplicates(DataTransform):
    """Remove duplicate rows and reset the row index."""

    def apply(self, df):
        return df.drop_duplicates().reset_index(drop=True)


class StandardizeText(DataTransform):
    """Trim whitespace and lowercase values in text columns."""

    def apply(self, df):
        result = df.copy()
        text_columns = result.select_dtypes(include="object").columns
        for column in text_columns:
            result[column] = result[column].str.strip().str.lower()
        return result


class TransformFactory:
    """Create transformations from a descriptive name."""

    _transforms = {
        "normalize": NormalizeColumns,
        "remove_duplicates": RemoveDuplicates,
        "standardize_text": StandardizeText,
    }

    @classmethod
    def create(cls, name):
        key = name.strip().lower()
        try:
            return cls._transforms[key]()
        except KeyError as error:
            available = ", ".join(sorted(cls._transforms))
            raise ValueError(
                f"Unknown transformation '{name}'. Available: {available}"
            ) from error


# Example usage
clean_data = DataCleaner(FillMean()).clean(reader.read())
normalized_data = TransformFactory.create("normalize").apply(clean_data)
standardized_data = TransformFactory.create("standardize_text").apply(clean_data)
print(normalized_data.head())


Starting read...
Finished read.
read took 0.001589 seconds.
Starting clean...
Finished clean.
clean took 0.001849 seconds.
         id     name      age  height_cm  weight_kg         city     score
0  0.000000    Alice  0.25000   0.333333   0.433333     New York  0.588235
1  0.166667      Bob  0.40625   0.800000   0.500000  Los Angeles  0.882353
2  0.333333  Charlie  1.00000   0.533333   0.566667      Chicago  0.647059
3  0.500000    David  0.40625   0.466667   0.833333      Houston  0.000000
4  0.666667      Eva  0.00000   0.000000   0.000000     New York  0.764706


/tmp/ipykernel_54039/2109088169.py:40: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  text_columns = result.select_dtypes(include="object").columns


## Exercise 5: Build a Full Cleaning Pipeline (Template Method Pattern)

### Objective:
- Combine all previous concepts into one cleaning pipeline.
- Use the Template Method Pattern to define a consistent workflow.

### Concept:
The Template Method defines the skeleton of a process and lets subclasses override specific steps.

### Instructions:
1. Create an abstract class DataPipeline with a run() method defining steps:
- load()
- clean()
- transform()
- save()
2. Create CSVDataPipeline that implements each step using previous classes

In [ ]:
class DataPipeline(ABC):
    """Template for a data-cleaning workflow."""

    def run(self):
        self.load()
        self.clean()
        self.transform()
        self.save()
        return self.data

    @abstractmethod
    def load(self):
        pass

    @abstractmethod
    def clean(self):
        pass

    @abstractmethod
    def transform(self):
        pass

    @abstractmethod
    def save(self):
        pass


class CSVDataPipeline(DataPipeline):
    """Clean and transform a CSV file, then save the result."""

    def __init__(self, input_path, output_path, strategy, transformation):
        self.reader = CSVReader(input_path)
        self.cleaner = DataCleaner(strategy)
        self.transformation = transformation
        self.output_path = output_path
        self.data = None

    def load(self):
        self.data = self.reader.read()
        return self.data

    def clean(self):
        self.data = self.cleaner.clean(self.data)
        return self.data

    def transform(self):
        self.data = self.transformation.apply(self.data)
        return self.data

    def save(self):
        self.data.to_csv(self.output_path, index=False)
        return self.output_path


# Example usage: one run executes the fixed workflow in DataPipeline.
pipeline = CSVDataPipeline(
    input_path="sample_data (2).csv",
    output_path="cleaned_data.csv",
    strategy=FillMean(),
    transformation=TransformFactory.create("normalize"),
)
result = pipeline.run()
print(result.head())
